In [1]:
from typing_extensions import TypedDict
from typing import List, Literal
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

# llm = init_chat_model(model="llama3.1:8B", model_provider="ollama")
llm = init_chat_model(model="gpt-4o-mini", model_provider="openai")

In [2]:
class State(TypedDict):
    document: str
    summary: str
    sentiment: str
    key_points: str
    recommendation: str
    final_analysis: str

In [3]:
def get_summary(state: State):
    response = llm.invoke(
        f"Write a 3-sentence summary of this document {state["document"]}"
    )

    return {"summary": response.content}


def get_sentiment(state: State):
    response = llm.invoke(
        f"Analyse the sentiment and tone of this document {state["document"]}"
    )

    return {"sentiment": response.content}


def get_key_points(state: State):
    response = llm.invoke(
        f"List the 5 most important points of this document {state["document"]}"
    )

    return {"key_points": response.content}


def get_recommendation(state: State):
    response = llm.invoke(
        f"Based on the document, list 3 recommended next steps {state["document"]}"
    )

    return {"recommendation": response.content}


def get_final_analysis(state: State):
    response = llm.invoke(
        f"""
    Give me an analysis of the following report

    DOCUMENT ANALYSIS REPORT
    ========================

    EXECUTIVE SUMMARY:
    {state['summary']}

    SENTIMENT ANALYSIS:
    {state['sentiment']}

    KEY POINTS:
    {state.get("key_points", "")}

    RECOMMENDATIONS:
    {state.get('recommendation', "N/A")}
    """
    )

    return {"final_analysis": response.content}

In [4]:
graph_builder = StateGraph(State)

graph_builder.add_node("get_summary", get_summary)
graph_builder.add_node("get_sentiment", get_sentiment)
graph_builder.add_node("get_key_points", get_key_points)
graph_builder.add_node("get_recommendation", get_recommendation)
graph_builder.add_node("get_final_analysis", get_final_analysis)

graph_builder.add_edge(START, "get_summary")
graph_builder.add_edge(START, "get_sentiment")
graph_builder.add_edge(START, "get_key_points")
graph_builder.add_edge(START, "get_recommendation")

graph_builder.add_edge("get_summary", "get_final_analysis")
graph_builder.add_edge("get_sentiment", "get_final_analysis")
graph_builder.add_edge("get_key_points", "get_final_analysis")
graph_builder.add_edge("get_recommendation", "get_final_analysis")

graph_builder.add_edge("get_final_analysis", END)

graph = graph_builder.compile()

# graph

In [5]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()
    for chunk in graph.stream(
        {"document": document},
        stream_mode="updates", # 어떤 노드가 업데이트 했는지 볼 수 있음.
    ):
        print(chunk, "\n")
        

{'get_summary': {'summary': 'In a recent statement, the Federal Reserve expressed commitment to achieving maximum employment and stable prices, acknowledging an uptick in the unemployment rate and a slowdown in job gains, alongside elevated inflation levels. The Federal Open Market Committee opted to lower the federal funds rate by a quarter percentage point while also reducing securities holdings, citing a shift in the balance of risks to the economy, particularly in the labor market. Recent economic indicators revealed moderated growth in consumer spending, alongside a notable decline in job creation, causing policymakers to reassess the appropriate stance of monetary policy in light of these evolving conditions.'}} 

{'get_recommendation': {'recommendation': "Based on the provided document, here are three recommended next steps:\n\n1. **Continued Monitoring of Economic Indicators**: The Federal Open Market Committee (FOMC) should continuously assess incoming data related to employme